# Level 5: Oracles and the Deutsch-Jozsa Algorithm
## Q# Edition

In this notebook, you will:

1. Build **quantum oracles** in Q#
2. Implement the **Deutsch-Jozsa algorithm**
3. See how Q# makes oracle construction elegant with **functors** and **operations**

In [ ]:
import qsharp
from qsharp_widgets import Circuit
print("Q# ready!")

---
## 1. Quantum Oracles in Q#

An oracle implements: $U_f |x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$

Using the **phase kickback** trick: $U_f |x\rangle|{-}\rangle = (-1)^{f(x)} |x\rangle|{-}\rangle$

Let's build several oracle types:

In [ ]:
%%qsharp

// Constant oracle: f(x) = 0 (do nothing)
operation ConstantZeroOracle(input : Qubit[], output : Qubit) : Unit {
    // f(x) = 0 means no change to output qubit
}

// Constant oracle: f(x) = 1 (always flip)
operation ConstantOneOracle(input : Qubit[], output : Qubit) : Unit {
    X(output);
}

// Balanced oracle: f(x) = x_0 (first qubit)
operation BalancedOracle1(input : Qubit[], output : Qubit) : Unit {
    CNOT(input[0], output);
}

// Balanced oracle: f(x) = x_0 XOR x_1
operation BalancedOracle2(input : Qubit[], output : Qubit) : Unit {
    CNOT(input[0], output);
    CNOT(input[1], output);
}

---
## 2. The Deutsch-Jozsa Algorithm

**Problem**: Given a black-box $f$, determine if it's constant or balanced in **one** query.

**Algorithm**:
1. Initialize: $|0\rangle^{\otimes n}|1\rangle$
2. Apply H to all qubits
3. Query the oracle
4. Apply H to input qubits
5. Measure: all zeros = constant; otherwise = balanced

In [ ]:
%%qsharp

operation DeutschJozsa(
    n : Int,
    oracle : (Qubit[], Qubit) => Unit
) : Bool {
    // Returns true if constant, false if balanced
    use input = Qubit[n];
    use output = Qubit();

    // Step 1: Set output to |1>
    X(output);

    // Step 2: Apply H to all qubits
    for q in input { H(q); }
    H(output);

    // Step 3: Apply oracle
    oracle(input, output);

    // Step 4: Apply H to input qubits
    for q in input { H(q); }

    // Step 5: Measure input qubits
    mutable isConstant = true;
    for q in input {
        if M(q) == One {
            set isConstant = false;
        }
    }

    // Clean up
    ResetAll(input);
    Reset(output);

    return isConstant;
}

In [ ]:
# Test with constant oracle f(x) = 0
result = qsharp.eval("DeutschJozsa(3, ConstantZeroOracle)")
print(f"Constant zero oracle: isConstant = {result} ✓")

# Test with constant oracle f(x) = 1
result = qsharp.eval("DeutschJozsa(3, ConstantOneOracle)")
print(f"Constant one oracle:  isConstant = {result} ✓")

# Test with balanced oracle f(x) = x_0
result = qsharp.eval("DeutschJozsa(3, BalancedOracle1)")
print(f"Balanced oracle 1:    isConstant = {result} ✓")

# Test with balanced oracle f(x) = x_0 XOR x_1
result = qsharp.eval("DeutschJozsa(3, BalancedOracle2)")
print(f"Balanced oracle 2:    isConstant = {result} ✓")

print("\nAll correct in a SINGLE query each!")

In [ ]:
# Visualize the circuit for the balanced oracle
Circuit(qsharp.circuit("DeutschJozsa(3, BalancedOracle1)"))

### Running Statistically to Confirm Determinism

Unlike probabilistic algorithms, Deutsch-Jozsa gives the **same answer every time**:

In [ ]:
%%qsharp

operation DJManyRuns(
    n : Int,
    oracle : (Qubit[], Qubit) => Unit,
    shots : Int
) : (Int, Int) {
    mutable trueCount = 0;
    mutable falseCount = 0;

    for _ in 0..shots - 1 {
        let result = DeutschJozsa(n, oracle);
        if result { set trueCount += 1; }
        else { set falseCount += 1; }
    }

    return (trueCount, falseCount);
}

In [ ]:
print("Running each oracle 1000 times...")
print(f"Constant(0):  {qsharp.eval('DJManyRuns(3, ConstantZeroOracle, 1000)')}")
print(f"Constant(1):  {qsharp.eval('DJManyRuns(3, ConstantOneOracle, 1000)')}")
print(f"Balanced(x₀): {qsharp.eval('DJManyRuns(3, BalancedOracle1, 1000)')}")
print(f"Balanced(x₀⊕x₁): {qsharp.eval('DJManyRuns(3, BalancedOracle2, 1000)')}")
print("\nDeterministic: 100% correct every time!")

---
## 3. Key Takeaways

| Concept | Description |
|---------|-------------|
| **Oracle** | Black-box quantum operation |
| **Phase kickback** | $(-1)^{f(x)}$ encodes answer in phases |
| **Deutsch-Jozsa** | Exponential speedup: 1 query vs 2^(n-1)+1 |
| **Q# advantage** | Operations as first-class values for oracles |

---
## 4. Challenges

1. **New balanced oracle**: Write a balanced oracle for $f(x) = x_0 \oplus x_1 \oplus x_2$ and test it.
2. **Scaling test**: Run Deutsch-Jozsa with 5, 8, and 10 input qubits. The answer is always in 1 query!
3. **Bernstein-Vazirani**: Modify the oracle to encode a secret string $s$ where $f(x) = s \cdot x$ (bitwise dot product). Can you recover $s$?

In [ ]:
%%qsharp

// Your challenge code here!

---
**Next up: [Level 6 — Grover's Algorithm](../Level_06_Grovers_Algorithm/Level_06_QSharp.ipynb)**